# CHSA Medical Triage Agent - Colab Serving

Hybrid notebook for serving the CHSA POC from a Colab GPU runtime. The notebook handles setup, secrets, smoke checks, endpoint tests, and debug cells, while `scripts/serve_colab.py` remains the tested source of truth for launching vLLM and FastAPI.

Use a GPU runtime. Keep the serving cell running while testing the API.


## Runtime checks

Verify Python, current path, and GPU visibility before installing serving dependencies.


In [3]:
import os
import subprocess
import sys
from pathlib import Path

print("python=", sys.version)
print("cwd=", Path.cwd())
try:
    subprocess.run(["nvidia-smi"], check=False)
except FileNotFoundError:
    print("nvidia-smi not found; switch Colab to a GPU runtime before model serving.")

python= 3.12.13 (main, May  4 2026, 21:09:48) [Clang 22.1.3 ]
cwd= /home/nhkp/dev/OpenClassrooms/Projet_14/notebooks
nvidia-smi not found; switch Colab to a GPU runtime before model serving.


## Clone or update repository

This keeps the notebook usable from a fresh Colab runtime. If you already opened it inside the repo from VSCode, the cell simply pulls the latest changes.


In [4]:
import importlib.util

REPO_URL = "https://github.com/Nhkp/medical-triage-agent.git"
if importlib.util.find_spec("google.colab"):
    DEFAULT_REPO_DIR = Path("/content/medical-triage-agent")
else:
    DEFAULT_REPO_DIR = Path.cwd() / ".colab_runtime" / "medical-triage-agent"

REPO_DIR = DEFAULT_REPO_DIR

if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
elif Path.cwd().name == "medical-triage-agent" and (Path.cwd() / ".git").exists():
    REPO_DIR = Path.cwd()
    subprocess.run(["git", "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print("repo=", Path.cwd())

fatal: could not create leading directories of '/content/medical-triage-agent': Permission denied


CalledProcessError: Command '['git', 'clone', 'https://github.com/Nhkp/medical-triage-agent.git', '/content/medical-triage-agent']' returned non-zero exit status 128.

## Install serving dependencies

Colab already owns the CUDA runtime. Install serving packages without creating a project `.venv`.


In [ ]:
!python -m pip install -q -U fastapi uvicorn vllm pyngrok huggingface_hub

## Load Hugging Face credentials

Priority order: `.env`, Colab Secrets, then interactive prompt. The token is only stored in the process environment.


In [ ]:
from getpass import getpass


def load_dotenv(path: Path) -> dict[str, str]:
    values = {}
    if not path.exists():
        return values
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        values[key.strip()] = value.strip().strip(chr(34)).strip(chr(39))
    return values


dotenv = load_dotenv(Path(".env"))
token = dotenv.get("HF_TOKEN") or os.environ.get("HF_TOKEN")

if not token:
    try:
        from google.colab import userdata
        from google.colab.userdata import SecretNotFoundError, TimeoutException

        token = userdata.get("HF_TOKEN")
    except (ImportError, KeyError, SecretNotFoundError, TimeoutException):
        token = None

if not token:
    token = getpass("HF_TOKEN: ")

os.environ["HF_TOKEN"] = token
for key in ("HF_DPO_MODEL_REPO", "HF_SFT_MODEL_REPO", "VLLM_MODEL_ID"):
    if dotenv.get(key):
        os.environ[key] = dotenv[key]

print("HF token loaded:", bool(os.environ.get("HF_TOKEN")))

## Select model repo

Prefer the DPO adapter repo, then SFT adapter repo, then the base Qwen model for smoke serving.


In [ ]:
BASE_MODEL_REPO = os.environ.get("VLLM_BASE_MODEL_ID") or "Qwen/Qwen3-1.7B-Base"
ADAPTER_REPO = (
    os.environ.get("HF_DPO_MODEL_REPO")
    or os.environ.get("HF_SFT_MODEL_REPO")
    or os.environ.get("VLLM_LORA_ADAPTER")
    or "Lokhidor/medical-triage-qwen3-dpo-lora"
)
LORA_NAME = os.environ.get("VLLM_MODEL_ID") or "medical-triage-dpo"
os.environ["VLLM_BASE_MODEL_ID"] = BASE_MODEL_REPO
os.environ["VLLM_LORA_ADAPTER"] = ADAPTER_REPO
os.environ["VLLM_MODEL_ID"] = LORA_NAME
print("BASE_MODEL_REPO=", BASE_MODEL_REPO)
print("ADAPTER_REPO=", ADAPTER_REPO)
print("LORA_NAME=", LORA_NAME)

## Dry-run serving commands

This does not load the model. It prints the exact vLLM and FastAPI commands the script will launch.


In [ ]:
!python scripts/serve_colab.py --base-model "$BASE_MODEL_REPO" --adapter "$ADAPTER_REPO" --lora-name "$LORA_NAME" --dry-run

## Launch vLLM + FastAPI + ngrok

Run this cell and keep it running. Stop it with `Ctrl+C` when finished. Remove `--ngrok` if you only need to test inside Colab.


In [ ]:
!python scripts/serve_colab.py --base-model "$BASE_MODEL_REPO" --adapter "$ADAPTER_REPO" --lora-name "$LORA_NAME" --ngrok

## Local endpoint tests

Use these if you launched without ngrok, or from another Colab cell while the serving cell is running.


In [ ]:
!curl -s http://127.0.0.1:8080/health

In [ ]:
!curl -s -X POST http://127.0.0.1:8080/triage \
  -H "Content-Type: application/json" \
  -d '{"symptoms":["douleur thoracique","difficulte respiratoire"]}'

## Audit lookup

Paste an `audit_id` returned by `/triage`. Audit output must remain metadata-only.


In [ ]:
AUDIT_ID = "audit_REPLACE_ME"
!curl -s http://127.0.0.1:8080/audit/$AUDIT_ID

## Robustness and latency checks

Run these after the API is up. They write outputs under `outputs/evaluations`, which is ignored by git.


In [ ]:
!python scripts/evaluate_robustness.py --url http://127.0.0.1:8080
!python scripts/evaluate_latency.py --url http://127.0.0.1:8080 --iterations 12

## Debug: processes and ports

Use these cells if a previous run was interrupted or a port is already busy.


In [ ]:
!ps -ef | grep -E 'vllm|uvicorn|serve_colab' | grep -v grep || true

In [ ]:
import socket

for port in (8000, 8080):
    sock = socket.socket()
    result = sock.connect_ex(("127.0.0.1", port))
    print(port, "open" if result == 0 else "closed")
    sock.close()

In [ ]:
# Use only if you need to clean up stuck serving processes.
# !pkill -f 'vllm.entrypoints.openai.api_server' || true
# !pkill -f 'uvicorn medical_triage_agent.api:create_app' || true
# !pkill -f 'scripts/serve_colab.py' || true
